In [1]:

import scipy.ndimage as ndi
import skimage as ski
import glob
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np


# Quantify Files

In [21]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import time

cluster = SLURMCluster(
    cores=1,                      # Number of cores per job
    memory="16GB",                # Memory per job
    queue="compute",        # Queue/partition name
    walltime="22:00:00",          # Job time limit
    local_directory="$TMPDIR",    # Temporary directory (optional)
    log_directory="logs",          # Directory for log files (optional)
)

client = Client(cluster)
print(client)

cluster.scale(jobs=512)  # Request 4 jobs
client.cluster

<Client: 'tcp://10.0.53.5:38343' processes=0 threads=0, memory=0 B>


/home/smc/mambaforge/envs/dask_cellposesam/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41839 instead
  warnings.warn(


Dashboard: http://10.0.53.5:41839/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.0.53.5:38343,Workers: 0
Dashboard: http://10.0.53.5:41839/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [22]:
import dask
import cv2
import numpy as np

def backsub(inp, radius=60):
    filterSize =(radius, radius)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)

    return rtn

def dask_sub(inp, radius=60):
    # Build a list of delayed plane tasks and compute them locally
    img = np.reshape(inp, (-1, inp.shape[-2], inp.shape[-1]))
    # tasks = [dask.delayed(backsub)(i, radius) for i in img]
    # results = dask.compute(*tasks)  # local compute via wrapper's thread pool
    results = np.array([backsub(i, radius) for i in img])
    rtn = np.reshape(np.array(results), inp.shape)
    return rtn

In [23]:
fnames = glob.glob('concatenated/*.tif')
fnames

['concatenated/G15_montage.tif',
 'concatenated/I10_montage.tif',
 'concatenated/C20_montage.tif',
 'concatenated/I14_montage.tif',
 'concatenated/N12_montage.tif',
 'concatenated/F10_montage.tif',
 'concatenated/N15_montage.tif',
 'concatenated/F14_montage.tif',
 'concatenated/L24_montage.tif',
 'concatenated/B20_montage.tif',
 'concatenated/L03_montage.tif',
 'concatenated/K24_montage.tif',
 'concatenated/L18_montage.tif',
 'concatenated/E16_montage.tif',
 'concatenated/F13_montage.tif',
 'concatenated/M21_montage.tif',
 'concatenated/O01_montage.tif',
 'concatenated/H04_montage.tif',
 'concatenated/O12_montage.tif',
 'concatenated/O24_montage.tif',
 'concatenated/P05_montage.tif',
 'concatenated/D03_montage.tif',
 'concatenated/A08_montage.tif',
 'concatenated/J17_montage.tif',
 'concatenated/G07_montage.tif',
 'concatenated/I18_montage.tif',
 'concatenated/I03_montage.tif',
 'concatenated/G16_montage.tif',
 'concatenated/K09_montage.tif',
 'concatenated/L04_montage.tif',
 'concaten

In [24]:
def process_fname(fname, viewer=None):
    img = ski.io.imread(fname)
    img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
    img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
    img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)

    labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels.tiff'))
    labels = ski.transform.resize(labels, img.shape[:-1], order=0)
    mask = labels > 0

    ski.io.imsave(fname.replace('_montage.tif', '_montage_ds.tiff'), img.astype(np.uint16))
    ski.io.imsave(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'), labels.astype(np.uint16))

    if viewer is not None:
        viewer.add_image(img, channel_axis=-1)
    cdf = pd.DataFrame({'fname':fname, 'time':np.arange(img.shape[0]), 'c0':img[:,:,:,0].sum(axis=(1,2)), 'c1':img[:,:,:,1].sum(axis=(1,2)),
                        'mask':mask.sum(axis=(1,2)), 'masked_c0':(img[:,:,:,0]*mask).sum(axis=(1,2)), 'masked_c1':(img[:,:,:,1]*mask).sum(axis=(1,2))})
    return cdf

In [25]:
df = []
for idx, fname in enumerate(fnames):
    print(idx/len(fnames), fname)
    cdf = dask.delayed(process_fname)(fname)
    df.append(cdf)
df = dask.compute(*df)

0.0 concatenated/G15_montage.tif
0.002770083102493075 concatenated/I10_montage.tif
0.00554016620498615 concatenated/C20_montage.tif
0.008310249307479225 concatenated/I14_montage.tif
0.0110803324099723 concatenated/N12_montage.tif
0.013850415512465374 concatenated/F10_montage.tif
0.01662049861495845 concatenated/N15_montage.tif
0.019390581717451522 concatenated/F14_montage.tif
0.0221606648199446 concatenated/L24_montage.tif
0.024930747922437674 concatenated/B20_montage.tif
0.027700831024930747 concatenated/L03_montage.tif
0.030470914127423823 concatenated/K24_montage.tif
0.0332409972299169 concatenated/L18_montage.tif
0.036011080332409975 concatenated/E16_montage.tif
0.038781163434903045 concatenated/F13_montage.tif
0.04155124653739612 concatenated/M21_montage.tif
0.0443213296398892 concatenated/O01_montage.tif
0.04709141274238227 concatenated/H04_montage.tif
0.04986149584487535 concatenated/O12_montage.tif
0.05263157894736842 concatenated/O24_montage.tif
0.055401662049861494 concatenat

In [26]:
df = pd.concat(df)
df.to_csv('Quantification.csv')

In [27]:
cluster.close()
client.close()

In [13]:
df

,fname,time,c0,c1,mask,masked_c0,masked_c1
0,concatenated/G15_montage.tif,0,5.779567e+07,9.925691e+05,20345,3.934278e+07,99220.689941
1,concatenated/G15_montage.tif,1,6.338889e+07,1.017835e+06,19673,4.417325e+07,107426.965576
2,concatenated/G15_montage.tif,2,6.638511e+07,1.058759e+06,17655,4.414913e+07,115580.437256
3,concatenated/G15_montage.tif,3,7.204891e+07,1.191480e+06,16913,4.629793e+07,131276.251221
4,concatenated/G15_montage.tif,4,1.214311e+08,1.426806e+06,34589,1.045137e+08,264344.903320
...,...,...,...,...,...,...,...
11,concatenated/E08_montage.tif,11,1.080114e+06,1.144469e+06,18184,2.942096e+04,52441.412842
12,concatenated/E08_montage.tif,12,1.051457e+06,1.153063e+06,17183,2.798817e+04,50079.209473
13,concatenated/E08_montage.tif,13,1.057595e+06,1.143725e+06,18137,2.732175e+04,53005.464111
14,concatenated/E08_montage.tif,14,1.052235e+06,1.136286e+06,18000,2.813415e+04,48864.400146
